# Re-select + France filter (three submissions)
Inputs: challenge data, the predict output(s) (the v4.1 R job, or both per-country jobs), ber-v4-S, ber-v4-T.
Writes submission_base/ (as predicted), submission_base_fr/ (+ France filter), submission_shift_fr/ (test-shift thresholds + France filter). All must print PASS.

In [ ]:
import os
os.makedirs('/kaggle/working/src', exist_ok=True)
%cd /kaggle/working

In [ ]:
%%writefile src/__init__.py
# package (a cell magic needs a non-empty body)


In [ ]:
%%writefile src/select.py
"""Per-S1 set selection maximizing expected F0.5.

F0.5 for one S1 with k predictions, TP hits and T true matches: 1.25*TP / (k + 0.25*T)
(1 if k = T = 0). With calibrated candidate probabilities q (after the one-to-one rule),
E[F] for predicting the top-k is approximated by 1.25*E[TP_k] / (k + 0.25*E[T]),
and the empty set scores P(T = 0) = prod(1 - q). We pick the best k per S1.
"""
import numpy as np
import pandas as pd


def one_to_one(s1_idx, pool_idx, p):
    df = pd.DataFrame({"s1": s1_idx, "pool": pool_idx, "p": p})
    best = df.groupby("pool").p.transform("max")
    return df[df.p >= best].drop_duplicates("pool")


def expected_f_select(s1_idx, pool_idx, p, miss_prior=0.0, min_p=0.0, power=1.0):
    """miss_prior: expected number of true matches not among candidates (blocking misses) per S1."""
    df = one_to_one(s1_idx, pool_idx, p)
    q_all = df.p.values ** power
    df = df.assign(q=q_all).sort_values(["s1", "q"], ascending=[True, False])
    g = df.s1.values
    q = df.q.values
    first = np.r_[True, g[1:] != g[:-1]]
    starts = np.flatnonzero(first)
    grp = np.cumsum(first) - 1
    k = np.arange(len(g)) - starts[grp] + 1
    csum = np.cumsum(q)
    etp = csum - np.r_[0, csum][starts][grp]
    et = np.add.reduceat(q, starts)[grp] + miss_prior
    ef = 1.25 * etp / (k + 0.25 * et)
    logq0 = np.add.reduceat(np.log1p(-np.clip(q, 0, 1 - 1e-7)), starts)
    p_empty = np.exp(logq0) * np.exp(-miss_prior)
    # best k per group
    best_ef = pd.Series(ef).groupby(grp).transform("max").values
    kbest = pd.Series(np.where(ef >= best_ef, k, 10 ** 9)).groupby(grp).transform("min").values
    choose = (k <= kbest) & (best_ef[...] > p_empty[grp]) & (q >= min_p)
    return df[choose][["s1", "pool", "p"]]

In [ ]:
%%writefile reselect.py
"""Stitch per-country predict outputs and re-select with thresholds adjusted for the test decoy rate.

Test has ~5.8 pool records per US/India S1 vs ~4.7 in train; the extra records are negatives that land in the
mid-probability band, so the holdout-tuned threshold is too permissive on test. Per country:
  m_b = (test pairs per S1 in p-bin b - holdout positives per S1 in b) / holdout negatives per S1 in b
(best-S1-per-pool pairs only), then the threshold that maximizes holdout macro F0.5 with every holdout negative
in bin b counted m_b times. France (no labels) uses the pooled US+India holdout as reference.
Writes two submissions: <out>/base (the model's own selection) and <out>/shift (per-country adjusted thresholds).

  python scripts/reselect.py --scores <F-job work dirs...> --train-work <T or T2 work dir> --test-work <S work dir>
      --gt <data>/train/train_ground_truth.tsv --cands <one F-job output dir> --out <dir>"""
import argparse
import os
import pickle
import shutil

import numpy as np
import pandas as pd

BINS = np.array([0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.001])
THRS = np.round(np.arange(0.50, 0.951, 0.025), 3)


def s1_hash(ids):
    return (pd.util.hash_array(np.asarray(ids, dtype=object)) % 10_000).astype(np.int32)


def best_per_pool(s1, pool, p):
    d = pd.DataFrame({"s1": s1, "pool": pool, "p": p})
    d = d[d.p > 0]
    d = d.sort_values("p", ascending=False, kind="stable").drop_duplicates("pool")
    return d


def per_s1_bins(d, n_s1, col=None):
    """pairs per S1 in each p bin (optionally only rows where col is True/False)."""
    b = np.digitize(d.p.values, BINS) - 1
    ok = (b >= 0) & (b < len(BINS) - 1)
    return np.bincount(b[ok], minlength=len(BINS) - 1) / max(n_s1, 1)


def weighted_f05(h, k_true, s1_all, thr, m):
    """holdout macro F0.5 when each negative in bin b counts m[b] times."""
    sel = h[h.p >= thr]
    b = np.clip(np.digitize(sel.p.values, BINS) - 1, 0, len(BINS) - 2)
    w = np.where(sel.y.values > 0, 0.0, m[b])
    tp = pd.Series(sel.y.values.astype(float)).groupby(sel.s1.values).sum()
    fp = pd.Series(w).groupby(sel.s1.values).sum()
    k = k_true.reindex(s1_all).fillna(0).values
    tpv = tp.reindex(s1_all).fillna(0).values
    fpv = fp.reindex(s1_all).fillna(0).values
    npred = tpv + fpv
    f = np.where(npred == 0, np.where(k == 0, 1.0, 0.0), 1.25 * tpv / np.maximum(tpv + fpv + 0.25 * k, 1e-9))
    f = np.where((k == 0) & (npred > 0), 0.0, f)
    return f.mean()


def write_lists(path, s1_ids, s1_idx, pool_ids, pool_idx, col):
    order = np.lexsort((pool_idx, s1_idx))
    s1_idx, pool_idx = s1_idx[order], pool_idx[order]
    bounds = np.searchsorted(s1_idx, np.arange(len(s1_ids) + 1))
    with open(path, "w") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for r in range(len(s1_ids)):
            f.write(f"{s1_ids[r]}\t{','.join(pool_ids[pool_idx[bounds[r]:bounds[r + 1]]])}\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scores", nargs="+", required=True, help="work dirs of the predict jobs (test_scores.parquet)")
    ap.add_argument("--train-work", required=True, help="dir with holdout_preds.parquet + model.pkl (+ train_norm_s1)")
    ap.add_argument("--train-s1", default=None, help="train_norm_s1.parquet if not in --train-work")
    ap.add_argument("--test-work", required=True, help="dir with test_norm_s1.parquet + test_norm_pool.parquet")
    ap.add_argument("--gt", required=True)
    ap.add_argument("--cands", required=True, help="dir with candidate_pairs.tsv (copied to both submissions)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--max-raise", type=float, default=0.15)
    a = ap.parse_args()

    # merged test scores (each predict job holds p for its countries, 0 elsewhere; same candidate order)
    sc = None
    for d in a.scores:
        x = pd.read_parquet(os.path.join(d, "test_scores.parquet"))
        if sc is None:
            sc = x
        else:
            assert len(x) == len(sc) and (x.s1_idx.values == sc.s1_idx.values).all(), "different candidate sets"
            sc["p"] = np.maximum(sc.p.values, x.p.values)
    t1 = pd.read_parquet(os.path.join(a.test_work, "test_norm_s1.parquet"), columns=["entity_id", "country"])
    tp_ids = np.asarray(pd.read_parquet(os.path.join(a.test_work, "test_norm_pool.parquet"), columns=["entity_id"]).entity_id.values, dtype=object)
    t_ids = np.asarray(t1.entity_id.values, dtype=object)
    t_ct = np.asarray(t1.country.values, dtype=object)
    M = pickle.load(open(os.path.join(a.train_work, "model.pkl"), "rb"))
    method = M.get("method", ("threshold", M["thr"]))
    base_thr = round(float(M["thr"]), 3)
    print("model selection:", method, "| holdout threshold", base_thr)

    # holdout with labels and S1 countries
    hp = pd.read_parquet(os.path.join(a.train_work, "holdout_preds.parquet"))
    tr_path = a.train_s1 or os.path.join(a.train_work, "train_norm_s1.parquet")
    tr = pd.read_parquet(tr_path, columns=["entity_id", "country"])
    tr_ids, tr_ct = np.asarray(tr.entity_id.values, dtype=object), np.asarray(tr.country.values, dtype=object)
    gt = pd.read_csv(a.gt, sep="\t", dtype=str, keep_default_na=False, quoting=3)
    k_all = pd.Series(gt.matched_entity_ids.str.count("S[23]-").values, gt.source1_entity_id.values)
    hs1 = np.unique(hp.s1_idx.values)
    h = best_per_pool(hp.s1_idx.values, hp.pool_idx.values, hp.p.values).rename(columns={})
    h["y"] = hp.set_index(["s1_idx", "pool_idx"]).y.reindex(pd.MultiIndex.from_arrays([h.s1.values, h.pool.values])).values
    h_ct = tr_ct[h.s1.values]
    k_true = pd.Series(k_all.reindex(tr_ids[hs1]).fillna(0).values, index=hs1)

    t = best_per_pool(sc.s1_idx.values, sc.pool_idx.values, sc.p.values)
    t_ctry = t_ct[t.s1.values]
    thr_by = {}
    for ct in pd.unique(t_ct):
        ref = [c for c in pd.unique(tr_ct) if c == ct] or list(pd.unique(tr_ct))     # unseen country: pooled holdout
        hm = np.isin(h_ct, ref)
        hs = hs1[np.isin(tr_ct[hs1], ref)]
        hh = h[hm]
        pos = per_s1_bins(hh[hh.y > 0], len(hs))
        neg = per_s1_bins(hh[hh.y == 0], len(hs))
        n_neg = np.bincount(np.clip(np.digitize(hh[hh.y == 0].p.values, BINS) - 1, 0, len(BINS) - 2), minlength=len(BINS) - 1)
        n_t = int((t_ct == ct).sum())
        tot = per_s1_bins(t[t_ctry == ct], n_t)
        m = np.where(n_neg >= 20, np.clip((tot - pos) / np.maximum(neg, 1e-9), 0.5, 5.0), 1.0)
        f_plain = {thr: weighted_f05(hh, k_true, hs, thr, np.ones_like(m)) for thr in THRS}
        f_shift = {thr: weighted_f05(hh, k_true, hs, thr, m) for thr in THRS}
        t_plain = max(f_plain, key=f_plain.get)
        t_shift = max(f_shift, key=f_shift.get)
        t_use = min(max(t_shift, base_thr), base_thr + a.max_raise)
        thr_by[ct] = t_use
        print(f"{ct}: {n_t:,} S1 | ref {ref} | decoy multiplier by bin {np.round(m, 2).tolist()}")
        print(f"   holdout-best thr {t_plain} | shift-aware thr {t_shift} (weighted F0.5 {f_shift[t_shift]:.4f} vs "
              f"{f_shift.get(base_thr, float('nan')):.4f} at {base_thr}) -> using {t_use}")

    for name, rule in (("base", None), ("shift", thr_by)):
        od = os.path.join(a.out, name)
        os.makedirs(od, exist_ok=True)
        if rule is None and method[0] == "expected_f":
            import sys
            sys.path.insert(0, os.getcwd())
            from src.select import expected_f_select
            sel = expected_f_select(sc.s1_idx.values, sc.pool_idx.values, sc.p.values, miss_prior=method[1][0], power=method[1][1])
            s_s1, s_pool = sel.s1.values, sel.pool.values
        else:
            thr_row = np.full(len(t), float(method[1]) if rule is None else 0.0)
            if rule is not None:
                thr_row = np.array([rule[c] for c in t_ctry])
            keep = t.p.values >= thr_row
            s_s1, s_pool = t.s1.values[keep], t.pool.values[keep]
        write_lists(os.path.join(od, "matching_results.tsv"), t_ids, np.asarray(s_s1, np.int64), tp_ids,
                    np.asarray(s_pool, np.int64), "matched_entity_ids")
        shutil.copy(os.path.join(a.cands, "candidate_pairs.tsv"), os.path.join(od, "candidate_pairs.tsv"))
        cnt = pd.Series(t_ct[np.asarray(s_s1, np.int64)]).value_counts() / pd.Series(t_ct).value_counts()
        print(f"{name}: {len(s_s1):,} matched pairs; per S1 by country {cnt.round(3).to_dict()}")


if __name__ == "__main__":
    main()

In [ ]:
import glob, sys, subprocess
I = "/kaggle/input"
def dirs(name): return sorted({os.path.dirname(p) for p in glob.glob(f"{I}/**/{name}", recursive=True)})
t1 = glob.glob(f"{I}/**/test/test_source1.tsv", recursive=True); assert t1, "attach the challenge data"
DATA = os.path.dirname(os.path.dirname(t1[0]))
scores = [d for d in dirs("test_scores.parquet") if os.path.isfile(f"{d}/test_scores.parquet") and not os.path.islink(f"{d}/test_scores.parquet")]
import hashlib
md5 = lambda f: hashlib.md5(open(f, "rb").read()).hexdigest()
trainw = [d for d in dirs("holdout_preds.parquet") if os.path.isfile(f"{d}/model.pkl") and not os.path.islink(f"{d}/holdout_preds.parquet")]
used = {md5(f"{d}/model.pkl") for d in scores if os.path.exists(f"{d}/model.pkl")}
if used:
    trainw = [d for d in trainw if md5(f"{d}/model.pkl") in used] or trainw
    assert len(used) == 1, "the two predict jobs used different models"
real = lambda f: os.path.isfile(f) and not os.path.islink(f)       # skip symlinks saved by earlier jobs
tests1 = [d for d in dirs("test_norm_s1.parquet") if real(f"{d}/test_norm_s1.parquet") and real(f"{d}/test_norm_pool.parquet")]
trs1 = [f for f in glob.glob(f"{I}/**/train_norm_s1.parquet", recursive=True) if real(f)]
cands = [os.path.dirname(p) for p in glob.glob(f"{I}/**/output/candidate_pairs.tsv", recursive=True)]
print("scores:", scores, "\ntrain:", trainw, "\ntest:", tests1, "\ncands:", cands[:1])
assert len(scores) in (1, 2), f"attach one predict output (R job) or both per-country outputs; found {scores}"
assert trainw and tests1 and trs1 and cands
r = subprocess.run([sys.executable, "reselect.py", "--scores", *scores, "--train-work", trainw[0], "--train-s1", trs1[0],
                    "--test-work", tests1[0], "--gt", f"{DATA}/train/train_ground_truth.tsv", "--cands", cands[0],
                    "--out", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])

In [ ]:
%%writefile france_filter.py
"""Post-processing for the unseen country: drop matched pairs whose pool name ADDS a decoy marker word.

Train evidence (dev candidates, close pairs): a name that gains the country name ('india'), 'group' or 'holdings'
is a true copy 0.0% of the time (9.5k, 19.9k, 16.6k pairs), with or without a dropped word. The French test
equivalents are 'france', 'groupe', 'holding(s)', 'participations', 'developpement'; the v3 stem fallback only
partly catches them (v3.1 still accepts ~10k French pairs that add one). Generic suffixes that true copies do
add ('services', 'associes', 'fils', 'cie', legal forms) are NOT touched.

  python scripts/france_filter.py --matching <in matching_results.tsv> --test-dir <data>/test --out <out.tsv>
Plain Python; streams the files."""
import argparse
import re
import unicodedata
from collections import Counter

WORDS = {"france", "groupe", "holding", "holdings", "participations", "participation", "developpement"}
TOK = re.compile(r"[a-z0-9]+")


def toks(s):
    return set(TOK.findall(unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().lower()))


def names(path, keep_country):
    out = {}
    with open(path, encoding="utf-8", newline="") as f:
        head = f.readline().rstrip("\r\n").split("\t")
        i_id, i_n, i_c = head.index("entity_id"), head.index("business_name"), head.index("country")
        for line in f:
            p = line.rstrip("\r\n").split("\t")
            if len(p) > max(i_id, i_n, i_c) and p[i_c] in keep_country:
                out[p[i_id]] = p[i_n]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--matching", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--countries", default="France")
    ap.add_argument("--words", default=",".join(sorted(WORDS)))
    a = ap.parse_args()
    keep = {c.strip() for c in a.countries.split(",")}
    words = {w.strip() for w in a.words.split(",") if w.strip()}
    s1 = names(f"{a.test_dir}/test_source1.tsv", keep)
    pool = names(f"{a.test_dir}/test_source2.tsv", keep)
    pool.update(names(f"{a.test_dir}/test_source3.tsv", keep))
    s1_toks = {k: toks(v) for k, v in s1.items()}
    removed, kept, rows_changed = Counter(), 0, 0
    with open(a.matching, encoding="utf-8", newline="") as f, open(a.out, "w", encoding="utf-8", newline="") as g:
        g.write(f.readline())
        for line in f:
            sid, _, m = line.rstrip("\r\n").partition("\t")
            if sid not in s1_toks or not m:
                g.write(line if line.endswith("\n") else line + "\n")
                continue
            base = s1_toks[sid]
            out = []
            for p in m.split(","):
                hit = (toks(pool.get(p, "")) - base) & words
                if hit:
                    removed.update(hit)
                else:
                    out.append(p)
                    kept += 1
            rows_changed += len(out) != len(m.split(","))
            g.write(f"{sid}\t{','.join(out)}\n")
    print(f"{a.countries}: kept {kept:,} pairs, removed {sum(removed.values()):,} marker hits in {rows_changed:,} S1 rows: {dict(removed)}")


if __name__ == "__main__":
    main()


In [ ]:
import shutil
for src, dst in (("base", "base_fr"), ("shift", "shift_fr")):
    os.makedirs(f"/kaggle/working/{dst}", exist_ok=True)
    r = subprocess.run([sys.executable, "france_filter.py", "--matching", f"/kaggle/working/{src}/matching_results.tsv",
                        "--test-dir", f"{DATA}/test", "--out", f"/kaggle/working/{dst}/matching_results.tsv"], capture_output=True, text=True)
    print(dst, r.stdout, r.stderr[-1000:])
    os.link(f"/kaggle/working/{src}/candidate_pairs.tsv", f"/kaggle/working/{dst}/candidate_pairs.tsv")
shutil.rmtree("/kaggle/working/shift")        # keep three submissions: base, base_fr, shift_fr

In [ ]:
V = (glob.glob(os.path.dirname(DATA) + "/utils/validate_submission.py") or glob.glob(f"{I}/**/validate_submission.py", recursive=True) or [None])[0]
for name in ("base", "base_fr", "shift_fr"):
    os.rename(f"/kaggle/working/{name}", f"/kaggle/working/submission_{name}") if os.path.exists(f"/kaggle/working/{name}") else None
    if V:
        r = subprocess.run([sys.executable, V, "--matching", f"/kaggle/working/submission_{name}/matching_results.tsv",
                            "--candidate", f"/kaggle/working/submission_{name}/candidate_pairs.tsv", "--test-dir", f"{DATA}/test"],
                           capture_output=True, text=True)
        print(name, r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])

In [ ]:
# sanity: submission_base must equal the union of the predict job(s)' own output/matching_results.tsv
def rows(f):
    with open(f, encoding="utf-8") as h:
        next(h)
        return {s: set(m.split(",")) - {""} for s, _, m in (l.rstrip("\n").partition("\t") for l in h)}
b, u = rows("/kaggle/working/submission_base/matching_results.tsv"), {}
owns = [os.path.join(os.path.dirname(d), "output", "matching_results.tsv") for d in scores]
for f in owns:
    if os.path.isfile(f):
        for s, m in rows(f).items():
            u.setdefault(s, set()).update(m)
if u and all(os.path.isfile(f) for f in owns):
    bad = sum(1 for s in set(u) | set(b) if u.get(s, set()) != b.get(s, set()))
    print(f"submission_base vs the predict job(s)' own output: {bad:,} S1 rows differ (must be 0)")